# MehuLLM — merge adapter → GGUF → Ollama

Takes the LoRA adapter from `02_lora_t4.ipynb` and produces a quantised GGUF
that runs on the GTX 1650.

### Why this is a separate notebook

Unsloth's built-in `save_pretrained_gguf` breaks whenever llama.cpp changes its
build layout. If that happens inside the training notebook you lose the training
run with it. Here, conversion is isolated and pinned to a known llama.cpp commit,
so a failed conversion costs minutes instead of hours.

Runs fine on a **CPU** Colab runtime — merging and quantising need RAM, not GPU.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

WORK = "/content/drive/MyDrive/mehullm"
ADAPTER = f"{WORK}/adapter"
MERGED = "/content/merged_fp16"  # local scratch: big, and we only keep the GGUF

assert os.path.exists(ADAPTER), f"{ADAPTER} missing -- run 02_lora_t4.ipynb first"
print(os.listdir(ADAPTER))

## 1 · Merge the adapter into the base weights

In [ ]:
# NOT the training lockfile. That pins unsloth + torchao 0.10, and PEFT's
# is_torchao_available() raises on that version -- which is the whole reason the
# merge below is hand-rolled. Merging needs transformers + safetensors only.
!pip install -q transformers accelerate safetensors sentencepiece protobuf


In [ ]:
# Manual LoRA merge: W' = W + (B @ A) * (alpha / r)
#
# NOT PeftModel.merge_and_unload(). PEFT's is_torchao_available() RAISES on an
# old torchao instead of returning False, so importing PeftModel is fatal on a
# runtime with torchao 0.10. The merge itself is two matmuls -- PEFT is not
# needed for it, and skipping PEFT skips the broken check entirely.
import collections
import json as _json
import re

import safetensors.torch as st
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "Qwen/Qwen3-1.7B"

cfg = _json.load(open(f"{ADAPTER}/adapter_config.json"))
scale = cfg["lora_alpha"] / cfg["r"]
print(f"r={cfg['r']} alpha={cfg['lora_alpha']} scale={scale}")

sd = st.load_file(f"{ADAPTER}/adapter_model.safetensors")
pairs = collections.defaultdict(dict)
for k, v in sd.items():
    m = re.match(r"^(?:base_model\.model\.)?(.+?)\.lora_([AB])(?:\.default)?\.weight$", k)
    if m:
        pairs[m.group(1)][m.group(2)] = v
print(f"{len(pairs)} LoRA target modules in the adapter")

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True
)
params = dict(base.named_parameters())

merged_n, missing = 0, []
for target, ab in pairs.items():
    if "A" not in ab or "B" not in ab:
        continue
    name = f"{target}.weight"
    if name not in params:
        alt = [p for p in params if p.endswith(name.split("model.", 1)[-1])]
        if not alt:
            missing.append(name)
            continue
        name = alt[0]
    W = params[name]
    delta = (ab["B"].float() @ ab["A"].float()) * scale
    if delta.shape != tuple(W.shape):
        missing.append(f"{name} {tuple(delta.shape)} vs {tuple(W.shape)}")
        continue
    with torch.no_grad():
        W += delta.to(W.dtype)
    merged_n += 1

print(f"merged {merged_n} modules")
if missing:
    print("UNMERGED:", missing[:5])
# 7 target modules x 28 layers. A silent 0 here would save an unmodified base
# model and the "fine-tune" would look like it simply did not work.
assert merged_n == 196, f"expected 196, got {merged_n}"

base.save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)
print("merged ->", MERGED)


## 2 · Build llama.cpp

Only the `llama-quantize` target is built — the converter is a plain Python
script and needs no compilation. Output is intentionally **not** captured:
a silent build failure here surfaces as a nonsense error two cells later.


In [ ]:
# NO %%capture here, deliberately. Hiding this cell's output is what turned a
# cmake failure into a confusing error three cells later.
#
# Tracking master rather than a pinned tag: the old pin's requirements.txt
# downgrades numpy and pulls a full torch, which is a bigger risk than a
# converter rename for a one-shot conversion. LLAMA_CURL=OFF because libcurl
# dev headers are the most common cmake failure on a fresh Colab image.
!rm -rf /content/llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q gguf sentencepiece protobuf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build \
    -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF -DBUILD_SHARED_LIBS=ON
!cmake --build /content/llama.cpp/build --config Release -j$(nproc) --target llama-quantize


In [ ]:
import glob
import os

conv = next(
    (p for p in ("/content/llama.cpp/convert_hf_to_gguf.py",
                 "/content/llama.cpp/convert-hf-to-gguf.py") if os.path.exists(p)),
    None,
)
quant = next(iter(glob.glob("/content/llama.cpp/build/bin/llama-quantize*")), None)
print("converter:", conv)
print("quantize :", quant)
assert conv and quant, "build failed -- scroll up through the previous cell for the cmake error"


## 3 · Convert and quantise

**Q4_K_M** is the target: ~1.1 GB of weights, plus KV cache at `num_ctx 2048`,
comfortably inside the 1650's measured 3.7 GB of free VRAM.

In [ ]:
import os

F16 = "/content/voice-f16.gguf"
Q4 = f"{WORK}/voice-q4km.gguf"

# Asserted after each step: `!` magics do not raise on a non-zero exit code, so
# without these a failed conversion falls through to a confusing getsize error.
!python {conv} {MERGED} --outtype f16 --outfile {F16}
assert os.path.exists(F16), "conversion failed -- read the converter output above"

!{quant} {F16} {Q4} Q4_K_M
assert os.path.exists(Q4), "quantisation failed -- read the output above"

print(f"\nf16 {os.path.getsize(F16) / 1e9:.2f} GB -> q4_k_m {os.path.getsize(Q4) / 1e9:.2f} GB")
print(f"saved to Drive: {Q4}")


## 4 · Modelfile

The `SYSTEM` line must match the system prompt used in training
(`neutralize.SYSTEM_PROMPT`) — a mismatch at serving time is a silent quality
regression that looks like a bad fine-tune.

`temperature 0.85` is deliberately high: this is a style task, and low
temperature produces flat, samey phrasing.

In [ ]:
# TEMPLATE is explicit and byte-matches notebook 02's `_prompt_text`. Without it
# Ollama falls back to the Qwen3 chat template embedded in the GGUF, whose
# handling of the empty <think> prefill is version-dependent -- a silent quality
# regression that looks like a bad fine-tune.
#
# temperature 0.3, NOT the 0.85 originally planned. Measured on v1: at 0.85 the
# same draft produced "Mai yek dumleye bolunga"; at 0.3, "I'll tell you by night
# / not sure yet". The style survives the drop, the coherence does not survive
# the rise.
MODELFILE = """FROM ./voice-q4km.gguf

PARAMETER num_ctx 2048
PARAMETER num_predict 200
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.05
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"

TEMPLATE \"\"\"<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
<think>

</think>

\"\"\"

SYSTEM \"\"\"Rewrite DRAFT as Mehul would send it on WhatsApp. Keep the meaning and all facts, names, numbers and links exactly. Match his length, script mix, punctuation and emoji habits. Output only the message.\"\"\"
"""

with open(f"{WORK}/Modelfile", "w", encoding="utf-8", newline="\n") as fh:
    fh.write(MODELFILE)
print(MODELFILE)


## 5 · Install locally

Download **`voice-q4km.gguf`** and **`Modelfile`** from `MyDrive/mehullm/` into a
folder on your laptop, then:

```powershell
cd <that folder>
ollama create mehul-voice -f Modelfile
ollama run mehul-voice "<context>\nPerson_A: kal aa raha hai?\n</context>\n<draft>\nI will let you know by tonight.\n</draft>"
```

Then set `OLLAMA_VOICE_MODEL=mehul-voice` in `.env`.

### Then the part that actually matters

```powershell
uv run mehullm-eval style --held-out
```

This scores the LoRA against **few-shot prompting on the base model**, on the two
chats withheld from training entirely. If the LoRA wins, that is your headline
result. If it does not, *"with N pairs, in-context style exemplars matched a
rank-32 LoRA on human preference"* is **also a result** — and you have a working
system either way.